# Experiment 0: second-campaign checks

Debugging evidence only: every pilot audit, the exact-resume, FP32-capacity and evaluation checks, monitor timing and the LGD density audit. Nothing here is a scientific result.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp, campaign2 as c2
from src.visualize.figures import FigureSaver
style.apply()
sink = FigureSaver('experiment0/01_campaign2_checks')
report = cp.NotebookReport('Experiment 0: second-campaign checks', sink=sink)

## 1. Pilot audits and GPU checks

Pass/fail per pilot and per check task.

In [2]:
summary = c2.experiment0_summary()
tables = c2.experiment0_tables(summary)
print('Experiment 0 summary available:', summary is not None)
for name, frame in tables.items():
    report.table(name, frame)
report.add('1. Pilot audits and GPU checks', 'The local output has no Experiment 0 summary receipt.' if summary is None else
           'Overall gate in the available receipt: ' + str(summary.get('passed')))

Experiment 0 summary available: False


## 2. LGD density roughness

Median adjacent-bucket log ratio of each original TabPFN regressor per LGD table.

In [3]:
import pandas as pd
density = (summary or {}).get('density_audit') or {}
report.table('Density roughness', pd.DataFrame(density.get('rows', [])))
report.add('2. LGD density roughness', 'Flagging threshold for the median adjacent-bucket log ratio: 2.5.' if density else
           'The local summary contains no density measurements.')

""


## 3. Training precision against FP32

The direction of the update can change even when two policies produce similar losses. The figure compares the gradient on each bounded episode with FP32. The tables identify the production policy recorded when each probe ran. Earlier TabPFN v2 PD probes used ordinary FP16 autocast. The revised policy keeps its transformer normalizations in FP32 under FP16 autocast. The full-episode table reports the selected production-size checks separately. Older timing-only measurements have no cosine.

In [4]:
from src.visualize import results2 as r2
agreement = c2.precision_agreement(summary)
fig = r2.precision_agreement(agreement, title='Mixed-precision gradients against FP32 at update zero')
cp.show(sink, c2.pages(fig, 'precision_agreement',
    'Cosine between the mixed-precision and FP32 gradients of the training loss at the pretrained weights, '
    'on identical episodes of up to 8,192 rows from partition 0. Each point represents a measured training '
    'table for the indicated task and model. The legend identifies the autocast policy. Both FP16 policies '
    'use loss scaling. The separate FP32-normalization policy applies to TabPFN v2 PD. TabICLv2 keeps '
    'its column embedder in FP32. A cosine of 1 indicates identical directions. The dashed line marks '
    'the 0.90 agreement threshold. Unavailable comparisons are listed in the table.'))
report.table('Bounded episodes: gradient agreement and the recorded production policy', agreement)
report.table('Full training episodes: recorded policy, gradient agreement, time and peak GPU memory',
             c2.precision_full_episodes(summary))
report.add('3. Training precision against FP32', c2.evidence_note('precision check', agreement))

No measurements available for this section yet.


""


""


## Summary

Every displayed table and figure, in section order.

In [5]:
print(report.summary(sink))

Experiment 0: second-campaign checks

1. Pilot audits and GPU checks
The local output has no Experiment 0 summary receipt.

2. LGD density roughness
The local summary contains no density measurements.

Table: Density roughness
CSV (first column: row index)
row

3. Training precision against FP32
No results yet for precision check; the figure is drawn once they exist.

No measurements available for this figure group.

Table: Bounded episodes: gradient agreement and the recorded production policy
CSV (first column: row index)
row

Table: Full training episodes: recorded policy, gradient agreement, time and peak GPU memory
CSV (first column: row index)
row

experiment0/01_campaign2_checks: no figures saved.
